## Feature Engineering & Desenvolvimento de Métricas

Este notebook tem como objetivo desenvolver features espaciais e métricas analíticas através da combinação dos dados de eventos (StatsBomb Event Data) com os dados contextuais disponibilizados pelo StatsBomb 360.

O foco desta fase será enriquecer a informação ao nível dos eventos com contexto espacial, permitindo construir métricas que suportem análises mais avançadas sobre jogadores e momentos do jogo.

Os datasets produzidos neste notebook serão posteriormente utilizados na construção do relatório final em Power BI.

In [266]:
# importações

import pandas as pd
import numpy as np
from math import sqrt

In [267]:
# carregamento dos datasets produzidos no notebook anterior

matches_df = pd.read_csv(
    'data/processed/matches.csv'
)

players_df = pd.read_csv(
    'data/processed/players.csv'
)

# se não fizer este passo, o pandas lê um csv com valores e infere float64, tenho de 'forçar' texto
players_df = pd.read_csv(
    'data/processed/players.csv',
    dtype={
        'player_id': 'string',
        'jersey_number': 'string',
        'position_id': 'string'
    }
)

events_df = pd.read_csv(
    'data/processed/events.csv'
)

# e a mesma coisa nos events
events_df = pd.read_csv(
    'data/processed/events.csv',
    dtype={
        'player_id': 'string',
        'team_id': 'string'
    }
)

frames_df = pd.read_csv(
    'data/processed/frames.csv'
)

In [268]:
# validação dos dados

print(f"Matches: {matches_df.shape}")
print(f"Players: {players_df.shape}")
print(f"Events: {events_df.shape}")
print(f"Frames: {frames_df.shape}")

Matches: (64, 16)
Players: (50, 13)
Events: (4407, 41)
Frames: (61451, 9)


****

## Modelo de Dados

O modelo de Dados definido durante a fase exploratória será mantido nesta fase do projeto.

<br>

**DIM_MATCHES**

Primary Key - match_id

<br>

**DIM_PLAYERS**

Primary Key - player_id

<br>

**FCT_EVENTS**

Primary Key - id (event_id)

Foreign Keys - match_id, player_id, team_id

<br>

**FCT_FRAMES**

Foreign Key - id (event_id)

****

## Métricas Candidatas

Antes da implementação das métricas finais, será efetuada uma fase de exploração com o objetivo de identificar quais as métricas mais relevantes para descrever o contexto espacial das ações dos jogadores.

### Features Base

As seguintes features servirão de base para o desenvolvimento das métricas finais:

- **nearest_opponent_distance**: distância entre o jogador associado ao evento e o adversário mais próximo presente no freeze frame;
- **opponents_within_5m**: número de adversários localizados num raio de 5 metros;
- **opponents_within_10m**: número de adversários localizados num raio de 10 metros;
- **available_teammates**: número de colegas de equipa potencialmente disponíveis como opção de passe.

### Métricas Finais

Com base nas features anteriores serão exploradas as seguintes métricas:

- **receiver pressure score**: mede o nível de pressão exercida sobre o jogador no momento em que recebe a bola;
- **passing options score**: avalia a quantidade de soluções de passe disponíveis para o jogador;
- **space availability score**: quantifica o espaço disponível em redor do jogador no momento da ação;
- **decision difficulty score**: procura estimar a dificuldade da decisão tomada, combinando fatores como pressão, espaço disponível e opções de passe.

Estas métricas procuram enriquecer os eventos com informação contextual, permitindo avaliar não apenas a ação realizada, mas também as condições em que essa ação ocorreu.

****

## Integração entre Event Data e 360 Data

O primeiro passo consiste em relacionar os eventos do jogo com a informação espacial disponibilizada pelos freeze frames.

Esta integração permitirá enriquecer os eventos com contexto espacial e servirá de base para o desenvolvimento das métricas apresentadas ao longo deste notebook.

Será criada uma tabela auxiliar que combinará Event Data e StatsBomb 360 Data.

Esta tabela terá como único objetivo suportar o desenvolvimento de features espaciais e métricas analíticas, não fazendo parte do modelo final utilizado na visualização.

In [269]:
# merge de events e frames

events_frames_df = (
    frames_df
    .merge(
        events_df,
        on=['id', 'match_id'],
        how='left'
    )
)

events_frames_df.shape

(61451, 48)

In [270]:
# confirmação de resultados

events_frames_df.head()

,id,match_id,teammate,actor,keeper,frame_location_x,frame_location_y,visible_area,visible_area_points,related_events,...,pass_outcome,pass_outswinging,pass_recipient,pass_shot_assist,pass_switch,pass_technique,pass_through_ball,pass_type,shot_outcome,shot_statsbomb_xg
0,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,39.216957,44.778613,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5,['97b5dc82-547a-4f93-a632-a2a8daf5ac98'],...,Complete,NaN,Aurélien Djani Tchouaméni,NaN,NaN,NaN,NaN,Kick Off,NaN,NaN
1,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,39.223561,29.496847,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5,['97b5dc82-547a-4f93-a632-a2a8daf5ac98'],...,Complete,NaN,Aurélien Djani Tchouaméni,NaN,NaN,NaN,NaN,Kick Off,NaN,NaN
2,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,42.576673,67.523741,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5,['97b5dc82-547a-4f93-a632-a2a8daf5ac98'],...,Complete,NaN,Aurélien Djani Tchouaméni,NaN,NaN,NaN,NaN,Kick Off,NaN,NaN
3,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,46.549120,12.613254,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5,['97b5dc82-547a-4f93-a632-a2a8daf5ac98'],...,Complete,NaN,Aurélien Djani Tchouaméni,NaN,NaN,NaN,NaN,Kick Off,NaN,NaN
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,3869685,True,False,False,47.798306,43.130706,"[8.98496759714251, 80.0, 41.4622037211361, 0.0...",5,['97b5dc82-547a-4f93-a632-a2a8daf5ac98'],...,Complete,NaN,Aurélien Djani Tchouaméni,NaN,NaN,NaN,NaN,Kick Off,NaN,NaN


In [271]:
# grande parte das métricas depende da distância entre os jogadores presentes no freezeframe e o jogador associado ao evento
# importa validar que cada evento possui um ator identificado

events_frames_df.groupby('id')['actor'].sum().value_counts()

1    3683
Name: actor, dtype: int64

****

## Distância ao Jogador Associado ao Evento

O dataset de frames regista não só o ator do evento (jogador que tem a bola), mas também outros visíveis na imagem da transmissão.

Para que consiga calcular a primeira feature, 'nearest_opponent_distance, preciso de calcular a distância de cada registo e posteriormente ficar apenas com a menor.

<br>

**1. Cálculo da feature**

In [272]:
events_frames_df['distance_to_event'] = np.sqrt(
    (
        events_frames_df['frame_location_x']
        -
        events_frames_df['location_x']
    ) ** 2
    +
    (
        events_frames_df['frame_location_y']
        -
        events_frames_df['location_y']
    ) ** 2
)

In [273]:
# confirmar o cálculo da feature para alguns registos

events_frames_df[
    [
        'id',
        'player',
        'team',
        'teammate',
        'actor',
        'distance_to_event'
    ]
].head()

,id,player,team,teammate,actor,distance_to_event
0,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,22.279820
1,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,24.220655
2,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,33.037562
3,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,31.053971
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,13.545106


In [274]:
# analisar a distribuição geral das distâncias calculadas

events_frames_df['distance_to_event'].describe()

count    61451.000000
mean        24.956936
std         18.094535
min          0.000000
25%         12.337360
50%         22.015829
75%         34.226350
max        134.201277
Name: distance_to_event, dtype: float64

In [275]:
# validar se os jogadores identificados como ator apresentam distância praticamente nula à localização do evento

events_frames_df[
    events_frames_df['actor'] == True
]['distance_to_event'].describe()

count    3.683000e+03
mean     4.561239e+00
std      1.863575e+01
min      0.000000e+00
25%      8.529922e-07
50%      1.572840e-06
75%      3.051758e-06
max      1.276386e+02
Name: distance_to_event, dtype: float64

Embora a mediana da distância observada para os jogadores identificados como ator seja praticamente nula, verificam-se alguns valores extremos superiores a 100 metros. Esta evidência sugere a existência de um conjunto reduzido de observações espacialmente inconsistentes, justificando uma análise mais detalhada.

<br>

**2. Análise da consistência espacial dos freeze frames**

Antes de utilizar a feature 'distance_to_event' como base para métricas espaciais mais avançadas, é necessário verificar se as coordenadas dos jogadores presentes nos freeze frames se encontram alinhadas com o sistema de coordenadas dos eventos.

Uma forma simples de realizar esta validação consiste em analisar os jogadores identificados como actor = 'True', uma vez que estes representam o jogador diretamente associado ao evento.

Em condições ideais, a posição do ator deverá coincidir ou aproximar-se da localização registada para o evento.

<br>

**3. Identificação de casos problemáticos**

In [276]:
problematicos = events_frames_df.loc[
    (events_frames_df["actor"] == True)
    & (events_frames_df["distance_to_event"] > 5)
].copy()

problematicos.head()

,id,match_id,teammate,actor,keeper,frame_location_x,frame_location_y,visible_area,visible_area_points,related_events,...,pass_outswinging,pass_recipient,pass_shot_assist,pass_switch,pass_technique,pass_through_ball,pass_type,shot_outcome,shot_statsbomb_xg,distance_to_event
172,6cb6421d-f26c-4193-95c5-fa22b7c69247,3869685,True,True,False,84.870432,3.696887,"[62.2002227830652, 80.0, 62.4318825346915, 5.8...",7,['8932a9c2-693a-4dde-afad-5115c7237e87'],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.112915
573,e1741082-3b80-48fa-b0e1-b979a3727cf1,3869685,True,True,False,75.462132,75.431648,"[51.9283836601778, 80.0, 56.3059706933406, 20....",5,['af7e1034-cf84-454b-bcad-7e8b8435d7b3'],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.839800
1127,5217dfc6-a682-488e-a33f-541138e26b5d,3869685,True,True,False,59.073437,73.599652,"[74.299848894901, 80.0, 39.5779956266617, 80.0...",5,['030b26ca-10a3-4d79-adfe-e8abe1762d07'],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.958459
1702,2caddad5-aa57-4afc-85d9-2ba269efa321,3869685,True,True,False,69.007795,60.840945,"[35.6193069982398, 80.0, 49.9607503995339, 13....",5,['5288bf75-ec36-47bc-a8e4-9568b39ddb0c'],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.682237
2236,a933f1bf-d0f1-461c-99f8-5101ba04b6f4,3869685,True,True,False,41.099998,20.299999,"[0.251344653035975, 80.0, 35.4191991294507, 0....",5,['8e21f64c-b0fd-4913-869e-9268d131722d'],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55.225449


O limiar de 5 metros foi escolhido por representar uma discrepância suficientemente elevada para não poder ser explicada apenas por arredondamentos, pequenas diferenças temporais ou imprecisão na captura dos freeze frames.

<br>

**4. Investigação dos casos problemáticos**

Reparei que algumas coordenadas, inexplicavelmente, parecem completamente simétricas ao registado em events.

Exemplo (tendo em conta que as coordenadas da statsbomb são 120, 80)
- Events - 116, 74
- Frames - 4, 6

Para estes casos, é possível pelo menos inverter os eixos nos registos de frames, de forma a coincidir com as coordenadas de events

In [277]:
# calcular o erro de alinhamento após inversão dos eixos

problematicos["x_flip_error"] = (
    problematicos["frame_location_x"]
    - (120 - problematicos["location_x"])
).abs()

problematicos["y_flip_error"] = (
    problematicos["frame_location_y"]
    - (80 - problematicos["location_y"])
).abs()

In [278]:
# calcular a distância ao evento após inversão da orientação do campo

problematicos["flip_distance"] = np.sqrt(
    (
        problematicos["frame_location_x"]
        - (120 - problematicos["location_x"])
    )**2
    +
    (
        problematicos["frame_location_y"]
        - (80 - problematicos["location_y"])
    )**2
)

In [279]:
problematicos[
    [
        'id',
        'x_flip_error',
        'y_flip_error',
        'flip_distance'
    ]
].head()

,id,x_flip_error,y_flip_error,flip_distance
172,6cb6421d-f26c-4193-95c5-fa22b7c69247,0.270432,0.903113,0.942733
573,e1741082-3b80-48fa-b0e1-b979a3727cf1,0.462132,0.768352,0.896622
1127,5217dfc6-a682-488e-a33f-541138e26b5d,0.973437,0.700348,1.199195
1702,2caddad5-aa57-4afc-85d9-2ba269efa321,1.307795,0.240945,1.329806
2236,a933f1bf-d0f1-461c-99f8-5101ba04b6f4,1.399998,0.500001,1.486606


<br>

**5. Resultados da Análise**

A análise revelou que uma parte significativa dos casos problemáticos pode ser explicada por diferenças de orientação espacial entre as coordenadas dos eventos e as coordenadas dos freeze frames.

Quando as coordenadas do evento são transformadas para a orientação oposta do campo (x'=120-x, y'=80-y), aproximadamente 77% dos casos problemáticos passam a apresentar uma discrepância inferior a 5 metros.

<br>

**6. Classificação dos casos problemáticos**

<p style="font-size:12px; font-weight:bold; margin:0;">Casos Corrigíveis:</p>

In [280]:
ids_corrigiveis = problematicos.loc[
    problematicos["flip_distance"] < 5,
    "id"
].unique()

In [281]:
ids_corrigiveis

array(['6cb6421d-f26c-4193-95c5-fa22b7c69247',
       'e1741082-3b80-48fa-b0e1-b979a3727cf1',
       '5217dfc6-a682-488e-a33f-541138e26b5d',
       '2caddad5-aa57-4afc-85d9-2ba269efa321',
       'a933f1bf-d0f1-461c-99f8-5101ba04b6f4',
       'c8c284b8-8db3-4f45-aaa6-eb85abbf6af0',
       '2ff8de78-7015-4416-a2e4-d17d5980f43c',
       '21f9f4e6-19b0-499c-b5de-b3761c856d5c',
       '13e66097-d801-433b-81af-1112c1008878',
       '58e36cd0-1a4c-4a35-ba4a-9fb0eee06bdb',
       '8b2e7f6b-d553-4f43-aed1-3450cbe9f434',
       'b42320b5-7276-4f76-a607-067802bda7af',
       '849c89a5-dc01-4f18-a486-8748d917bdb8',
       '6c8939e9-ca1a-43b6-825a-5024b3f1f48a',
       '72426e5a-996a-4ee2-91d6-3a95877aacc7',
       'f16c6887-843f-4aad-9a51-dd48e19dae46',
       'cc7c22cb-51d3-4cff-847d-96424358a67c',
       '3d4a3a82-a220-43a4-866f-c28cd32571f7',
       '0067de64-02ad-4a59-9bc1-9be587b18ea3',
       'bee3fdb0-edf5-41fa-8775-6f7d7fea7477',
       '31f4b99b-fd13-491f-8649-8e32fc6c45ee',
       '1f1f3

In [282]:
# exemplo de um id corrigivel

events_frames_df.loc[
    (events_frames_df["id"] == "6cb6421d-f26c-4193-95c5-fa22b7c69247")
    & (events_frames_df["actor"] == True),
    [
        "frame_location_x",
        "frame_location_y",
        "location_x",
        "location_y",
        "distance_to_event"
    ]
]

,frame_location_x,frame_location_y,location_x,location_y,distance_to_event
172,84.870432,3.696887,35.4,75.4,87.112915


<br>

<p style="font-size:12px; font-weight:bold; margin:0;">Casos Não Corrigíveis:</p>

In [283]:
ids_nao_corrigiveis = problematicos.loc[
    problematicos["flip_distance"] >= 5,
    "id"
].unique()

In [284]:
ids_nao_corrigiveis

array(['a5b72d1e-b750-4b67-8ecc-405d10ab46bd',
       '652e27c1-a3ad-4888-94bb-50da58cd1013',
       '5060d20d-d3a3-45e0-9328-12365e06dd25',
       '85b38871-bb92-4d24-80c0-d77795abda5f',
       'bf6ee042-2d47-4baf-8a11-38c81f2f8e80',
       '388a1908-c2f9-4898-9a10-ba3da12ba6c0',
       '109e5a19-6445-4deb-be7c-9311133952c3',
       '276d1403-1718-4be7-93b0-5239320ef911',
       '9594de8f-c1e6-4a85-ac02-b63c7f80ba56',
       '0243331e-182c-4634-b5d0-942c3e8ab24c',
       '0befcb3d-5ca5-4b0b-95f0-d7141bf0cbbc',
       '4a9bcd38-af58-4b78-980f-39abb359ad77',
       '776bc599-75ac-49cb-a586-a65fb77bcd54',
       '1eb565ec-14c3-4942-979a-037604ec250e',
       '4b964ca0-7b2a-4126-9996-223b7635494c',
       '7d913561-aeb9-46d1-99bc-12ef78185737',
       '8be43980-a270-49e5-bd3f-39dc72bbf514',
       '8e189c40-8174-441e-a688-163f7d7c88c3',
       '78d6352d-50b6-4693-9a09-fb16da954270',
       '4d0de017-e462-40c7-9642-e29985e86cd8',
       'a595070d-8355-4b38-8c7e-6c8d7598ac6f',
       'cf594

<br>

**7. Correção dos freeze frames**

In [285]:
# inversão das coordenadas

mask = events_frames_df["id"].isin(ids_corrigiveis)

events_frames_df.loc[mask, "frame_location_x"] = (
    120 - events_frames_df.loc[mask, "frame_location_x"]
)

events_frames_df.loc[mask, "frame_location_y"] = (
    80 - events_frames_df.loc[mask, "frame_location_y"]
)

Isto corrige:

- ator
- colegas
- adversários
- guarda-redes

em todo o freeze frame.

<br>

**8. Remoção dos casos não corrigíveis**

In [286]:
# quantificar a proporção de eventos classificados como não corrigíveis

events_frames_df["invalid_freeze_frame"] = (
    events_frames_df["id"].isin(ids_nao_corrigiveis)
)

events_frames_df["invalid_freeze_frame"].mean()

0.013181233828578868

Considerando que estes eventos representam uma fração muito reduzida da amostra total e que não foi possível identificar uma explicação consistente para as discrepâncias observadas, optou-se pela sua remoção antes do cálculo das restantes features espaciais.

In [287]:
n_removidos = events_frames_df["invalid_freeze_frame"].sum()
n_total = len(events_frames_df)

print(f"Registos a remover: {n_removidos}")
print(f"Registos totais: {n_total}")
print(f"Percentagem a remover: {n_removidos/n_total:.2%}")

Registos a remover: 810
Registos totais: 61451
Percentagem a remover: 1.32%


In [288]:
# remoção dos casos não corrigíveis

events_frames_df = events_frames_df.loc[
    ~events_frames_df["invalid_freeze_frame"]
].copy()

<br>

**CONCLUSÃO METODOLÓGICA**

A análise dos jogadores identificados como ator permitiu detetar um conjunto reduzido de eventos com inconsistências espaciais. 
- a maioria destes casos foi explicada por diferenças de orientação entre as coordenadas dos eventos e as coordenadas dos freeze frames, sendo corrigida através da inversão dos eixos do campo
- os restantes casos, para os quais não foi identificada uma explicação consistente, foram removidos da amostra de trabalho de forma a evitar a introdução de ruído nas métricas espaciais subsequentes

<br>

In [289]:
# recálculo da distância entre cada jogador presente no freeze frame e a localização do evento

events_frames_df['distance_to_event'] = np.sqrt(
    (
        events_frames_df['frame_location_x']
        -
        events_frames_df['location_x']
    ) ** 2
    +
    (
        events_frames_df['frame_location_y']
        -
        events_frames_df['location_y']
    ) ** 2
)

In [290]:
# confirmar o cálculo da feature para alguns registos

events_frames_df[
    [
        'id',
        'player',
        'team',
        'teammate',
        'actor',
        'distance_to_event'
    ]
].head()

,id,player,team,teammate,actor,distance_to_event
0,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,22.279820
1,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,24.220655
2,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,33.037562
3,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,31.053971
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,Antoine Griezmann,France,True,False,13.545106


In [291]:
# analisar a distribuição geral das distâncias calculadas

events_frames_df['distance_to_event'].describe()

count    60641.000000
mean        22.220303
std         14.169130
min          0.000000
25%         11.635592
50%         20.656040
75%         31.717587
max         78.452392
Name: distance_to_event, dtype: float64

In [292]:
# validar se os jogadores identificados como ator apresentam distância praticamente nula à localização do evento

events_frames_df[
    events_frames_df['actor'] == True
]['distance_to_event'].describe()

count    3.630000e+03
mean     8.470356e-02
std      4.570742e-01
min      0.000000e+00
25%      8.529922e-07
50%      1.572840e-06
75%      3.051758e-06
max      4.826629e+00
Name: distance_to_event, dtype: float64

Após a correção dos eventos com orientação invertida e a remoção dos casos não corrigíveis, a distância observada para os jogadores identificados como actor tornou-se praticamente nula em toda a amostra. O valor máximo registado é inferior a 5 metros, confirmando a consistência espacial do conjunto de dados utilizado nas etapas seguintes.

In [293]:
# teste ao mesmo exemplo de cima para confirmar que foi corrigido

events_frames_df.loc[
    (events_frames_df["id"] == "6cb6421d-f26c-4193-95c5-fa22b7c69247")
    & (events_frames_df["actor"] == True),
    [
        "frame_location_x",
        "frame_location_y",
        "location_x",
        "location_y",
        "distance_to_event"
    ]
]

,frame_location_x,frame_location_y,location_x,location_y,distance_to_event
172,35.129568,76.303113,35.4,75.4,0.942733


****

## CORREÇÃO DE FCT_FRAMES

Todas as alterações que foram feitas até agora foram apenas na events_frames_df que é o dataset que vou usar para calcular as métricas seguintes.

Mas o dataset frames_df ainda contém também estes erros e preciso de os corrigir porque vou usar mais tarde na visualização.

In [294]:
# confirmar valores com o mesmo exemplo anterior

frames_df.loc[
    frames_df["id"] == "6cb6421d-f26c-4193-95c5-fa22b7c69247",
    [
        "actor",
        "frame_location_x",
        "frame_location_y",
        "visible_area"
    ]
].head(20)

,actor,frame_location_x,frame_location_y,visible_area
162,False,62.915643,68.852107,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
163,False,67.380647,13.466581,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
164,False,70.168637,4.431013,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
165,False,71.171917,13.198251,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
166,False,71.640960,5.688987,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
167,False,74.840369,13.679771,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
168,False,83.115917,38.833844,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
169,False,83.435911,29.850736,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
170,False,84.243298,26.097452,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."
171,False,84.699997,4.700000,"[62.2002227830652, 80.0, 62.4318825346915, 5.8..."


In [295]:
# remoção de ids nao corrigíveis

frames_df = frames_df.loc[
    ~frames_df["id"].isin(ids_nao_corrigiveis)
].copy()

In [296]:
# corrigir frame_location_x e frame_location_y

mask = frames_df["id"].isin(ids_corrigiveis)

frames_df.loc[mask, "frame_location_x"] = (
    120 - frames_df.loc[mask, "frame_location_x"]
)

frames_df.loc[mask, "frame_location_y"] = (
    80 - frames_df.loc[mask, "frame_location_y"]
)

In [297]:
# é necessário também corrigir a coluna visible_area
# confirmar tipo de dados da coluna

type(frames_df["visible_area"].iloc[0])

str

In [298]:
# como é uma string preciso de converter para lista

import ast

type(ast.literal_eval(frames_df["visible_area"].iloc[0]))

list

In [299]:
# agora sim, posso corrigir a coluna de visible area

import ast

def flip_visible_area(coords):

    if isinstance(coords, str):
        coords = ast.literal_eval(coords)

    flipped = []

    for i in range(0, len(coords), 2):
        x = float(coords[i])
        y = float(coords[i + 1])

        flipped.extend([
            120 - x,
            80 - y
        ])

    return flipped

In [300]:
frames_df.loc[
    mask,
    "visible_area"
] = frames_df.loc[
    mask,
    "visible_area"
].apply(flip_visible_area)

In [301]:
# confirmar se mantenho a coluna em string

frames_df["visible_area"].head(3)

0    [8.98496759714251, 80.0, 41.4622037211361, 0.0...
1    [8.98496759714251, 80.0, 41.4622037211361, 0.0...
2    [8.98496759714251, 80.0, 41.4622037211361, 0.0...
Name: visible_area, dtype: object

In [302]:
type(frames_df["visible_area"].iloc[0])

str

In [303]:
# confirmar com o mesmo exemplo acima se as colunas estão corretas tal como em events_frames_df

frames_df.loc[
    frames_df["id"] == "6cb6421d-f26c-4193-95c5-fa22b7c69247",
    [
        "actor",
        "frame_location_x",
        "frame_location_y",
        "visible_area"
    ]
].head(20)

,actor,frame_location_x,frame_location_y,visible_area
162,False,57.084357,11.147893,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
163,False,52.619353,66.533419,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
164,False,49.831363,75.568987,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
165,False,48.828083,66.801749,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
166,False,48.359040,74.311013,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
167,False,45.159631,66.320229,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
168,False,36.884083,41.166156,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
169,False,36.564089,50.149264,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
170,False,35.756702,53.902548,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."
171,False,35.300003,75.300000,"[57.7997772169348, 0.0, 57.5681174653085, 74.1..."


****

## Cálculo das Features

- nearest_opponent_distance
- opponents_within_5m
- opponents_within_10m
- available_teammates



 **1. nearest_opponent_distance** - qual a distância ao adversário mais próximo do jogador associado ao evento

In [304]:
# considerar apenas adversários presentes no freeze frame

opponents_df = (
    events_frames_df[
        events_frames_df['teammate'] == False
    ]
    .copy()
)

In [305]:
# para cada evento, obter a menor distância a um adversário

nearest_opponent_distance = (
    opponents_df
    .groupby('id')['distance_to_event']
    .min()
    .reset_index()
)

In [306]:
# renomear a feature criada

nearest_opponent_distance = (
    nearest_opponent_distance
    .rename(
        columns={
            'distance_to_event':
            'nearest_opponent_distance'
        }
    )
)

In [307]:
# validar resultados

nearest_opponent_distance.head()

,id,nearest_opponent_distance
0,0018bfad-064a-47cb-8577-09b5dd716425,1.748244
1,001b4b48-a9ca-47e4-a805-7a12f0f09211,21.561630
2,00299352-83bb-4a9b-8f8e-39c6ff730340,2.024080
3,0043f0bc-1dc4-4841-8455-b84afcfef9f2,0.745483
4,00442448-fa73-427f-bf1d-9f4629e9a503,2.980686


In [308]:
# validar os valores obtidos para a feature criada

nearest_opponent_distance[
    'nearest_opponent_distance'
].describe()

count    3627.000000
mean        6.215245
std         5.545452
min         0.000002
25%         2.198364
50%         4.582194
75%         8.731904
max        45.961911
Name: nearest_opponent_distance, dtype: float64

In [309]:
# forma dos dados

nearest_opponent_distance.shape

(3627, 2)

In [310]:
# incorporar a feature na FCT_EVENTS

events_df = (
    events_df
    .merge(
        nearest_opponent_distance,
        on='id',
        how='left'
    )
)

In [311]:
# confirmar que a feature está na tabela

events_df[
    [
        'id',
        'nearest_opponent_distance'
    ]
].head()

,id,nearest_opponent_distance
0,0584ee21-e3dd-4d9f-95a0-5b5e84be25c3,NaN
1,b32679f8-942e-4122-96a2-015caf75e628,NaN
2,954f6855-de22-46a2-8d09-6fe94eec2b9b,NaN
3,6404a8e8-afaf-489d-b65e-173a237ffed5,NaN
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,9.155371


In [312]:
# validar os tipos de evento para os quais não foi possível calcular a feature

events_df[
    events_df['nearest_opponent_distance'].isna()
]['type'].value_counts()

Pass               268
Ball Receipt*      172
Carry               95
Pressure            60
Goal Keeper         23
Duel                18
Ball Recovery       15
Substitution        13
Foul Committed      12
Foul Won            12
Dribble             11
Half Start          10
Block               10
Half End            10
Injury Stoppage      9
Dispossessed         8
Tactical Shift       7
Dribbled Past        6
Miscontrol           5
Shot                 3
Clearance            3
Interception         2
50/50                2
Bad Behaviour        2
Starting XI          2
Player On            1
Player Off           1
Name: type, dtype: int64

<br>

**2. opponents_within_5m** - quantos adversários se encontram num raio de 5 metros do jogador associado ao evento

In [313]:
# contar adversários localizados até 5 metros do evento

opponents_within_5m = (
    opponents_df[
        opponents_df['distance_to_event'] <= 5
    ]
    .groupby('id')
    .size()
    .reset_index(
        name='opponents_within_5m'
    )
)

In [314]:
# analisar a distribuição da feature criada

opponents_within_5m[
    'opponents_within_5m'
].describe()

count    1943.000000
mean        1.319609
std         0.589351
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
Name: opponents_within_5m, dtype: float64

In [315]:
# incorporar a feature na FCT_EVENTS

events_df = (
    events_df
    .merge(
        opponents_within_5m,
        on='id',
        how='left'
    )
)

In [316]:
# confirmar que a feature está na tabela

events_df[
    [
        'id',
        'opponents_within_5m'
    ]
].head()

,id,opponents_within_5m
0,0584ee21-e3dd-4d9f-95a0-5b5e84be25c3,NaN
1,b32679f8-942e-4122-96a2-015caf75e628,NaN
2,954f6855-de22-46a2-8d09-6fe94eec2b9b,NaN
3,6404a8e8-afaf-489d-b65e-173a237ffed5,NaN
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,NaN


<br>

**3. opponents_within_10m** - quantos adversários se encontram num raio de 10 metros do jogador associado ao evento

In [317]:
# contar adversários localizados até 10 metros do evento

opponents_within_10m = (
    opponents_df[
        opponents_df['distance_to_event'] <= 10
    ]
    .groupby('id')
    .size()
    .reset_index(
        name='opponents_within_10m'
    )
)

In [318]:
# analisar a distribuição da feature criada

opponents_within_10m[
    'opponents_within_10m'
].describe()

count    2905.000000
mean        2.041308
std         1.138943
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         9.000000
Name: opponents_within_10m, dtype: float64

In [319]:
# incorporar a feature na FCT_EVENTS

events_df = (
    events_df
    .merge(
        opponents_within_10m,
        on='id',
        how='left'
    )
)

In [320]:
# confirmar que a feature está na tabela

events_df[
    [
        'id',
        'opponents_within_10m'
    ]
].head()

,id,opponents_within_10m
0,0584ee21-e3dd-4d9f-95a0-5b5e84be25c3,NaN
1,b32679f8-942e-4122-96a2-015caf75e628,NaN
2,954f6855-de22-46a2-8d09-6fe94eec2b9b,NaN
3,6404a8e8-afaf-489d-b65e-173a237ffed5,NaN
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,1.0


<br> 

In [321]:
# enquanto na primeira métrica NaN significava que não existe freeze frame
# aqui NaN significará 0 adversários encontrados até 5/10 metros
# por isso vou substituir NaN por zero para eventos sem adversários dentro dos raios considerados

events_df['opponents_within_5m'] = (
    events_df['opponents_within_5m']
    .fillna(0)
    .astype('Int64')
)

events_df['opponents_within_10m'] = (
    events_df['opponents_within_10m']
    .fillna(0)
    .astype('Int64')
)

<br>

**4. available_teammates** - quantos colegas de equipa se encontram num raio de 15 metros do jogador associado ao evento

In [322]:
# considerar apenas colegas de equipa presentes no freeze frame

teammates_df = (
    events_frames_df[
        events_frames_df['teammate'] == True
    ]
    .copy()
)

In [323]:
# excluir o jogador associado ao evento

teammates_df = (
    teammates_df[
        teammates_df['actor'] == False
    ]
)

In [324]:
# contar potenciais opções de passe até 15 metros

available_teammates = (
    teammates_df[
        teammates_df['distance_to_event'] <= 15
    ]
    .groupby('id')
    .size()
    .reset_index(
        name='available_teammates'
    )
)

In [325]:
# analisar a distribuição da feature criada

available_teammates[
    'available_teammates'
].describe()

count    2999.000000
mean        2.394465
std         1.467044
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000
Name: available_teammates, dtype: float64

In [326]:
# incorporar a feature na FCT_EVENTS

events_df = (
    events_df
    .merge(
        available_teammates,
        on='id',
        how='left'
    )
)

In [327]:
# aqui, ao contrário de nearest_opponent_distance, um NaN significa 0 opções de passe encontradas

events_df['available_teammates'] = (
    events_df['available_teammates']
    .fillna(0)
    .astype('Int64')
)

In [328]:
# confirmar que a feature está disponível

events_df[
    [
        'id',
        'available_teammates'
    ]
].head()

,id,available_teammates
0,0584ee21-e3dd-4d9f-95a0-5b5e84be25c3,0
1,b32679f8-942e-4122-96a2-015caf75e628,0
2,954f6855-de22-46a2-8d09-6fe94eec2b9b,0
3,6404a8e8-afaf-489d-b65e-173a237ffed5,0
4,f651a6c4-55e3-4e0f-a178-59414ba83d6a,2


In [329]:
# analisar a distribuição do número de opções de passe disponíveis

available_teammates['available_teammates'].describe()

count    2999.000000
mean        2.394465
std         1.467044
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000
Name: available_teammates, dtype: float64

In [330]:
# analisar conjuntamente as features espaciais desenvolvidas

events_df[
    [
        'nearest_opponent_distance',
        'opponents_within_5m',
        'opponents_within_10m',
        'available_teammates'
    ]
].describe()

,nearest_opponent_distance,opponents_within_5m,opponents_within_10m,available_teammates
count,3627.000000,4407.000000,4407.000000,4407.000000
mean,6.215245,0.581802,1.345587,1.629453
std,5.545452,0.763182,1.338412,1.646596
min,0.000002,0.000000,0.000000,0.000000
25%,2.198364,0.000000,0.000000,0.000000
50%,4.582194,0.000000,1.000000,1.000000
75%,8.731904,1.000000,2.000000,3.000000
max,45.961911,6.000000,9.000000,10.000000


<br>

**Resumo Cálculo Features**

Nem todos os eventos presentes no Event Data possuem freeze frame associado no StatsBomb 360.Consequentemente, algumas features espaciais apenas podem ser calculadas para um subconjunto dos eventos disponíveis.

Foram desenvolvidas quatro features espaciais que permitem caracterizar o contexto envolvente de cada evento:

- nearest_opponent_distance
- opponents_within_5m
- opponents_within_10m
- available_teammates

As features espaciais desenvolvidas permitem caracterizar o contexto envolvente de cada evento através da proximidade dos adversários e da disponibilidade de colegas de equipa.

Estas variáveis servirão de base para a construção de métricas compostas que procuram representar conceitos como pressão, espaço disponível e dificuldade de decisão.

****

## Cálculo das Métricas

- receiver pressure score
- passing options score
- space availability score
- decision difficulty score

**Abordagem para Construção dos Scores**

Após o desenvolvimento das features espaciais, o passo seguinte consiste na criação de métricas compostas que permitam representar conceitos como pressão, espaço disponível e dificuldade de decisão.

Existem diferentes abordagens possíveis para a construção deste tipo de métricas. Uma primeira alternativa consiste na normalização das variáveis com base na distribuição observada nos dados (por exemplo, utilizando técnicas como Min-Max Scaling). Embora esta abordagem permita obter scores comparáveis dentro da amostra analisada, apresenta uma limitação importante: os valores obtidos passam a depender diretamente do conjunto de jogos utilizado no cálculo.

Por exemplo, uma distância de 5 metros ao adversário mais próximo poderá originar scores diferentes consoante a distribuição das distâncias observadas em cada jogo ou competição. Desta forma, a interpretação dos resultados torna-se menos consistente quando se pretende aplicar o modelo a novos jogos ou expandir a análise para um conjunto de dados mais alargado.

Por esse motivo, optou-se por uma abordagem baseada em thresholds interpretáveis e ponderações fixas. Nesta abordagem, os scores são construídos a partir de regras inspiradas na interpretação do contexto futebolístico, permitindo associar níveis de pressão, espaço ou apoio a situações observáveis em campo.

Esta opção apresenta duas vantagens principais:

- mantém a consistência dos scores quando novos jogos são adicionados ao dataset;
- facilita a interpretação dos resultados, uma vez que cada componente do score possui um significado intuitivo e diretamente relacionado com o contexto de jogo.

As métricas desenvolvidas nas secções seguintes deverão, por isso, ser interpretadas como indicadores relativos de contexto espacial, construídos a partir de regras analíticas definidas para suportar a análise do jogo.

<br>

**1. RECEIVER PRESSURE SCORE** - mede o nível de pressão exercida sobre o jogador no momento em que recebe a bola

O receiver pressure score procura quantificar o nível de pressão exercida sobre o jogador associado ao evento.

Para o cálculo desta métrica são consideradas três componentes:

- distância ao adversário mais próximo
- número de adversários presentes num raio de 5 metros
- indicador de pressão disponibilizado pelo StatsBomb (`under_pressure`)

Cada componente é convertida para uma escala comum através de thresholds interpretáveis e posteriormente combinada utilizando ponderações fixas.

O resultado final varia entre 0 e 100, representando níveis crescentes de pressão exercida sobre o jogador no momento da ação.

Os thresholds utilizados em cada componente serão definidos com base na interpretação do contexto futebolístico e apresentados durante a implementação da métrica.

<br>

**Componente 1 — distância ao adversário mais próximo**

In [331]:
# converter a distância ao adversário mais próximo para uma escala entre 0 e 100
# quanto mais próximo do adversário, maior é a escala

def nearest_opponent_distance_score(distance):

    if pd.isna(distance):
        return np.nan

    elif distance <= 2:
        return 100

    elif distance <= 5:
        return 75

    elif distance <= 10:
        return 50

    elif distance <= 15:
        return 25

    else:
        return 0

In [332]:
# calcular score de proximidade ao adversário mais próximo

events_df['nearest_opponent_distance_score'] = (
    events_df['nearest_opponent_distance']
    .apply(nearest_opponent_distance_score)
)

In [333]:
# validar distribuição do score criado

events_df[
    'nearest_opponent_distance_score'
].describe()

count    3627.000000
mean       61.958919
std        29.628872
min         0.000000
25%        50.000000
50%        75.000000
75%        75.000000
max       100.000000
Name: nearest_opponent_distance_score, dtype: float64

<br>

**Componente 2 — adversários até 5 metros**

In [334]:
# converter o número de adversários próximos para uma escala entre 0 e 100
# quanto mais adversários tiver até 5m, maior é a escala

def opponents_within_5m_score(opponents):

    if pd.isna(opponents):
        return np.nan

    elif opponents >= 4:
        return 100

    elif opponents == 3:
        return 75

    elif opponents == 2:
        return 50

    elif opponents == 1:
        return 25

    else:
        return 0

In [335]:
# calcular score de densidade defensiva

events_df['opponents_within_5m_score'] = (
    events_df['opponents_within_5m']
    .apply(opponents_within_5m_score)
)

In [336]:
# validar distribuição do score criado

events_df[
    'opponents_within_5m_score'
].describe()

count    4407.000000
mean       14.522351
std        18.955173
min         0.000000
25%         0.000000
50%         0.000000
75%        25.000000
max       100.000000
Name: opponents_within_5m_score, dtype: float64

<br>

**Componente 3 — under pressure**

In [337]:
# converter indicador de pressão do StatsBomb para uma escala entre 0 a 100
# se há pressão 100, se não há 0

events_df['under_pressure_score'] = np.where(
    events_df['under_pressure'] == True,
    100,
    0
)

In [338]:
# validar distribuição do score criado

events_df[
    'under_pressure_score'
].describe()

count    4407.000000
mean       17.449512
std        37.957775
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: under_pressure_score, dtype: float64

<br>

**Combinar todos os componentes para calcular receiver_pressure_score**

In [339]:
# cálculo da métrica com diferentes thresholds

events_df['receiver_pressure_score'] = (
      events_df['nearest_opponent_distance_score'] * 0.50
    + events_df['opponents_within_5m_score'] * 0.30
    + events_df['under_pressure_score'] * 0.20
)

In [340]:
# analisar distribuição do receiver pressure score

events_df[
    'receiver_pressure_score'
].describe()

count    3627.000000
mean       39.824235
std        23.294155
min         0.000000
25%        25.000000
50%        45.000000
75%        57.500000
max       100.000000
Name: receiver_pressure_score, dtype: float64

In [341]:
# identificar eventos com maior nível de pressão
# quanto maior o score, mais pressão houve (porque todas as variáveis tendem a ter um maior score quanto mais pressão existe)

events_df[
    [
        'player',
        'team',
        'type',
        'receiver_pressure_score'
    ]
].sort_values(
    by='receiver_pressure_score',
    ascending=False
).head(10)

,player,team,type,receiver_pressure_score
3924,Lionel Andrés Messi Cuccittini,Argentina,Pass,100.0
4252,Ibrahima Konaté,France,Clearance,100.0
3925,Lionel Andrés Messi Cuccittini,Argentina,Ball Receipt*,100.0
3926,Lionel Andrés Messi Cuccittini,Argentina,Carry,100.0
3145,Randal Kolo Muani,France,Pass,100.0
880,Olivier Giroud,France,Clearance,100.0
4253,Nicolás Hernán Otamendi,Argentina,Duel,100.0
2669,Kingsley Coman,France,Dispossessed,92.5
803,Ángel Fabián Di María Hernández,Argentina,Dribble,92.5
801,Ángel Fabián Di María Hernández,Argentina,Carry,92.5


O Receiver Pressure Score combina proximidade dos adversários, densidade defensiva e informação contextual disponibilizada pelo StatsBomb para representar o nível de pressão exercido sobre o jogador associado ao evento.

Valores mais elevados indicam contextos de maior pressão, enquanto valores mais reduzidos sugerem situações com maior espaço e menor constrangimento defensivo.

<br>
<br>

**2. PASSING OPTIONS SCORE** - avalia a quantidade de soluções de passe disponíveis para o jogador

O passing options score procura quantificar a disponibilidade de soluções de passe para o jogador associado ao evento.

Para o cálculo desta métrica são consideradas três componentes:

- número de colegas de equipa disponíveis como opção de passe
- distância ao adversário mais próximo
- número de adversários presentes num raio de 5 metros

Cada componente é convertida para uma escala comum através de thresholds interpretáveis e posteriormente combinada utilizando ponderações fixas.

O resultado final varia entre 0 e 100, representando níveis crescentes de disponibilidade de opções de passe.

<br>

**Componente 1 — número de colegas de equipa disponíveis como opção de passe**

In [342]:
# converter o número de opções de passe para uma escala entre 0 e 100
# quanto mais opções de passe tem, maior é a escala

def available_teammates_score(teammates):

    if pd.isna(teammates):
        return np.nan

    elif teammates >= 4:
        return 100

    elif teammates == 3:
        return 75

    elif teammates == 2:
        return 50

    elif teammates == 1:
        return 25

    else:
        return 0

In [343]:
# calcular score de opções de passe disponíveis

events_df['available_teammates_score'] = (
    events_df['available_teammates']
    .apply(available_teammates_score)
)

In [344]:
# validar distribuição do score criado

events_df[
    'available_teammates_score'
].describe()

count    4407.000000
mean       38.019061
std        34.592519
min         0.000000
25%         0.000000
50%        25.000000
75%        75.000000
max       100.000000
Name: available_teammates_score, dtype: float64

<br>

**Componente 2 — distância ao adversário mais próximo**

In [345]:
# converter a distância ao adversário mais próximo para uma escala entre 0 e 100
# quanto maior é a distância para o adversário mais próximo maior é a escala

def available_space_score(distance):

    if pd.isna(distance):
        return np.nan

    elif distance <= 2:
        return 0

    elif distance <= 5:
        return 25

    elif distance <= 10:
        return 50

    elif distance <= 15:
        return 75

    else:
        return 100

In [346]:
# calcular score de espaço disponível

events_df['available_space_score'] = (
    events_df['nearest_opponent_distance']
    .apply(available_space_score)
)

In [347]:
# validar distribuição do score criado

events_df[
    'available_space_score'
].describe()

count    3627.000000
mean       38.041081
std        29.628872
min         0.000000
25%        25.000000
50%        25.000000
75%        50.000000
max       100.000000
Name: available_space_score, dtype: float64

<br> 

**Componente 3 — adversários até 5m**

In [348]:
# converter o número de adversários próximos para uma escala entre 0 e 100
# quanto menos adversários tiver até 5m, maior é a escala

def opponents_within_5m_score(opponents):

    if pd.isna(opponents):
        return np.nan

    elif opponents >= 4:
        return 0

    elif opponents == 3:
        return 25

    elif opponents == 2:
        return 50

    elif opponents == 1:
        return 75

    else:
        return 100

In [349]:
# calcular score de densidade defensiva

events_df['nearby_opponents_score'] = (
    events_df['opponents_within_5m']
    .apply(opponents_within_5m_score)
)

In [350]:
# validar distribuição do score criado

events_df[
    'nearby_opponents_score'
].describe()

count    4407.000000
mean       85.477649
std        18.955173
min         0.000000
25%        75.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: nearby_opponents_score, dtype: float64

<br> 

**Combinar todos os componentes para calcular passing_options_score**

In [351]:
# combinar as componentes através de ponderações fixas

events_df['passing_options_score'] = (
      events_df['available_teammates_score'] * 0.50
    + events_df['available_space_score'] * 0.30
    + events_df['nearby_opponents_score'] * 0.20
)

In [352]:
# analisar distribuição do passing options score

events_df[
    'passing_options_score'
].describe()

count    3627.000000
mean       50.980838
std        15.441467
min         5.000000
25%        40.000000
50%        50.000000
75%        60.000000
max        92.500000
Name: passing_options_score, dtype: float64

In [353]:
# identificar eventos com maior disponibilidade de opções de passe
# quanto maior o score, mais opções de passe (porque todas as variáveis tendem a ter um maior score quanto mais opções existe)

events_df[
    [
        'player',
        'team',
        'type',
        'passing_options_score'
    ]
].sort_values(
    by='passing_options_score',
    ascending=False
).head(10)

,player,team,type,passing_options_score
304,Enzo Fernandez,Argentina,Ball Recovery,92.5
2692,Hugo Lloris,France,Goal Keeper,92.5
3091,Raphaël Varane,France,Ball Receipt*,92.5
3092,Raphaël Varane,France,Carry,92.5
3941,Leandro Daniel Paredes,Argentina,Carry,92.5
3940,Leandro Daniel Paredes,Argentina,Ball Receipt*,92.5
303,Enzo Fernandez,Argentina,Carry,92.5
3706,Nicolás Alejandro Tagliafico,Argentina,Clearance,92.5
773,Adrien Rabiot,France,Ball Receipt*,92.5
3533,Nahuel Molina Lucero,Argentina,Pass,87.5


<br>
<br>

**3. SPACE AVAILABILITY SCORE** - quantifica o espaço disponível em redor do jogador no momento da ação

O space availability score procura quantificar o espaço disponível em redor do jogador associado ao evento.

Para o cálculo desta métrica são consideradas três componentes:

- distância ao adversário mais próximo
- número de adversários presentes num raio de 5 metros
- número de adversários presentes num raio de 10 metros

Cada componente é convertida para uma escala comum através de thresholds interpretáveis e posteriormente combinada utilizando ponderações fixas.

O resultado final varia entre 0 e 100, representando níveis crescentes de espaço disponível para o jogador no momento da ação.

<br>

**Componente 1 — distância ao adversário mais próximo**

In [354]:
# reutilizar a componente de espaço disponível baseada na distância ao adversário mais próximo,
# desenvolvida anteriormente durante o cálculo do Passing Options Score
# quanto maior é a distância para o adversário mais próximo maior é a escala

events_df[
    'available_space_score'
].describe()

count    3627.000000
mean       38.041081
std        29.628872
min         0.000000
25%        25.000000
50%        25.000000
75%        50.000000
max       100.000000
Name: available_space_score, dtype: float64

<br>

**Componente 2 — adversários até 5m**

In [355]:
# reutilizar a componente baseada na presença de adversários próximos,
# invertida anteriormente para representar disponibilidade de passe e espaço disponível
# quanto menos adversários tiver até 5m, maior é a escala

events_df[
    'nearby_opponents_score'
].describe()

count    4407.000000
mean       85.477649
std        18.955173
min         0.000000
25%        75.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: nearby_opponents_score, dtype: float64

<br>

**Componente 3 — adversários até 10m**

In [356]:
# converter o número de adversários num raio de 10 metros para uma escala entre 0 e 100
# quanto menos adversários tiver até 10m , maior é a escala

def opponents_within_10m_space_score(opponents):

    if pd.isna(opponents):
        return np.nan

    elif opponents >= 8:
        return 0

    elif opponents >= 6:
        return 25

    elif opponents >= 4:
        return 50

    elif opponents >= 2:
        return 75

    else:
        return 100

In [357]:
# calcular score de espaço disponível considerando adversários num raio de 10 metros

events_df['opponents_within_10m_space_score'] = (
    events_df['opponents_within_10m']
    .apply(opponents_within_10m_space_score)
)

In [358]:
# validar distribuição do score criado

events_df[
    'opponents_within_10m_space_score'
].describe()

count    4407.000000
mean       88.081461
std        16.301557
min         0.000000
25%        75.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: opponents_within_10m_space_score, dtype: float64

<br>

**Combinar todos os componentes para calcular space_availability_score**

In [359]:
# combinar as componentes através de ponderações fixas

events_df['space_availability_score'] = (
      events_df['available_space_score'] * 0.40
    + events_df['nearby_opponents_score'] * 0.40
    + events_df['opponents_within_10m_space_score'] * 0.20
)

In [360]:
# analisar distribuição do space availability score

events_df[
    'space_availability_score'
].describe()

count    3627.000000
mean       65.261924
std        20.626904
min         0.000000
25%        50.000000
50%        60.000000
75%        80.000000
max       100.000000
Name: space_availability_score, dtype: float64

In [361]:
# identificar eventos com maior disponibilidade de espaço
# quanto maior o score, mais espaço existe (porque todas as variáveis tender a ter um maior score quanto mais espaço existe)

events_df[
    [
        'player',
        'team',
        'type',
        'space_availability_score'
    ]
].sort_values(
    by='space_availability_score',
    ascending=False
).head(10)

,player,team,type,space_availability_score
2118,Jules Koundé,France,Pass,100.0
1997,Cristian Gabriel Romero,Argentina,Carry,100.0
1874,Jules Koundé,France,Ball Receipt*,100.0
1875,Jules Koundé,France,Carry,100.0
1885,Raphaël Varane,France,Ball Receipt*,100.0
1886,Raphaël Varane,France,Carry,100.0
1929,Damián Emiliano Martínez,Argentina,Pass,100.0
1930,Damián Emiliano Martínez,Argentina,Ball Receipt*,100.0
1931,Nicolás Hernán Otamendi,Argentina,Ball Receipt*,100.0
1932,Nicolás Hernán Otamendi,Argentina,Carry,100.0


<br>
<br>

**4. DECISION DIFFICULTY SCORE** - procura estimar a dificuldade da decisão tomada, combinando fatores como pressão, espaço disponível e opções de passe

O Decision Difficulty Score procura estimar a dificuldade associada à decisão tomada pelo jogador no momento do evento.

Para o cálculo desta métrica são consideradas três componentes:

- nível de pressão exercida sobre o jogador
- disponibilidade de opções de passe
- espaço disponível em redor do jogador

As métricas anteriormente desenvolvidas são reutilizadas nesta etapa, permitindo combinar diferentes dimensões do contexto espacial numa única medida sintética.

**MUITO IMPORTANTE!!**
As 3 métricas têm escalas de 0 a 100, mas nem todas representam a mesma coisa.
Enquanto a primeira, quanto mais perto de 100 maior é a dificuldade, as restantes duas tendem para 100 quando existe mais facilidade.
Por isso, vamos reutilizar as métricas sim, mas fazendo um pequeno twist mais abaixo.

O resultado final variará, posteriormente, entre 0 e 100, representando níveis crescentes de dificuldade de decisão.

<br>

**Componente 1 — Receiver Pressure Score**

In [362]:
# reutilizar a métrica de pressão desenvolvida anteriormente

events_df[
    'receiver_pressure_score'
].describe()

count    3627.000000
mean       39.824235
std        23.294155
min         0.000000
25%        25.000000
50%        45.000000
75%        57.500000
max       100.000000
Name: receiver_pressure_score, dtype: float64

<br>

**Componente 2 — Passing Options Score**

In [363]:
# na métrica anteriormente calculada um maior score indicava mais opções
# por isso é necessário inverter a escala de opções de passe para representar dificuldade de decisão

events_df['passing_options_difficulty_score'] = (
    100 -
    events_df['passing_options_score']
)

In [364]:
# validar distribuição da componente criada

events_df[
    'passing_options_difficulty_score'
].describe()

count    3627.000000
mean       49.019162
std        15.441467
min         7.500000
25%        40.000000
50%        50.000000
75%        60.000000
max        95.000000
Name: passing_options_difficulty_score, dtype: float64

<br>

**Componente 3 — Space Availability Score**

In [365]:
# na métrica anteriormente calculada um maior score indicava mais espaço
# por isso é necessário inverter a escala de espaço disponível para representar dificuldade de decisão

events_df['space_difficulty_score'] = (
    100 -
    events_df['space_availability_score']
)

In [366]:
# validar distribuição da componente criada

events_df[
    'space_difficulty_score'
].describe()

count    3627.000000
mean       34.738076
std        20.626904
min         0.000000
25%        20.000000
50%        40.000000
75%        50.000000
max       100.000000
Name: space_difficulty_score, dtype: float64

<br>

**Combinar todos os componentes para calcular decision_difficulty_score**

In [367]:
# combinar as componentes através de ponderações fixas

events_df['decision_difficulty_score'] = (
      events_df['receiver_pressure_score'] * 0.50
    + events_df['passing_options_difficulty_score'] * 0.25
    + events_df['space_difficulty_score'] * 0.25
)

In [368]:
# analisar distribuição do decision difficulty score

events_df[
    'decision_difficulty_score'
].describe()

count    3627.000000
mean       40.851427
std        18.122795
min         3.125000
25%        25.625000
50%        41.875000
75%        54.375000
max        90.000000
Name: decision_difficulty_score, dtype: float64

In [369]:
# identificar eventos associados a maior dificuldade de decisão

events_df[
    [
        'player',
        'team',
        'type',
        'decision_difficulty_score'
    ]
].sort_values(
    by='decision_difficulty_score',
    ascending=False
).head(10)

,player,team,type,decision_difficulty_score
2393,Alexis Mac Allister,Argentina,Dispossessed,90.000
3924,Lionel Andrés Messi Cuccittini,Argentina,Pass,89.375
3926,Lionel Andrés Messi Cuccittini,Argentina,Carry,89.375
3925,Lionel Andrés Messi Cuccittini,Argentina,Ball Receipt*,89.375
961,Lionel Andrés Messi Cuccittini,Argentina,Pass,86.875
3145,Randal Kolo Muani,France,Pass,86.250
2009,Aurélien Djani Tchouaméni,France,Pass,85.625
4253,Nicolás Hernán Otamendi,Argentina,Duel,85.000
880,Olivier Giroud,France,Clearance,85.000
4252,Ibrahima Konaté,France,Clearance,85.000


<br>

**AVALIAR CORELAÇÃO ENTRE MÉTRICAS**

Antes da definição final do Decision Difficulty Score, é importante efetuar uma análise de correlação entre as métricas desenvolvidas com o objetivo de compreender o grau de relação existente entre elas.

In [370]:
events_df[
    [
        'receiver_pressure_score',
        'passing_options_score',
        'space_availability_score',
        'decision_difficulty_score'
    ]
].corr()

,receiver_pressure_score,passing_options_score,space_availability_score,decision_difficulty_score
receiver_pressure_score,1.000000,-0.309994,-0.936434,0.975164
passing_options_score,-0.309994,1.000000,0.274408,-0.490318
space_availability_score,-0.936434,0.274408,1.000000,-0.944819
decision_difficulty_score,0.975164,-0.490318,-0.944819,1.000000


Esta análise permitiu validar se as diferentes métricas estavam a capturar dimensões distintas do contexto espacial ou se, pelo contrário, apresentavam comportamentos excessivamente semelhantes.

Observou-se uma forte correlação negativa entre o Receiver Pressure Score e o Space Availability Score, resultado esperado uma vez que ambas as métricas representam perspetivas opostas do mesmo fenómeno espacial: maior pressão tende a estar associada a menor espaço disponível.

<br>

In [371]:
#teste com 40 | 30 | 30

events_df['decision_difficulty_score'] = (
      events_df['receiver_pressure_score'] * 0.40
    + events_df['passing_options_difficulty_score'] * 0.30
    + events_df['space_difficulty_score'] * 0.30
)

In [372]:
#corelação entre métricas

events_df[
    [
        'receiver_pressure_score',
        'passing_options_score',
        'space_availability_score',
        'decision_difficulty_score'
    ]
].corr()

,receiver_pressure_score,passing_options_score,space_availability_score,decision_difficulty_score
receiver_pressure_score,1.000000,-0.309994,-0.936434,0.960155
passing_options_score,-0.309994,1.000000,0.274408,-0.534891
space_availability_score,-0.936434,0.274408,1.000000,-0.939048
decision_difficulty_score,0.960155,-0.534891,-0.939048,1.000000


<br>

In [373]:
#teste com 35 | 35 | 30

events_df['decision_difficulty_score'] = (
      events_df['receiver_pressure_score'] * 0.35
    + events_df['passing_options_difficulty_score'] * 0.35
    + events_df['space_difficulty_score'] * 0.30
)

In [374]:
events_df[
    [
        'receiver_pressure_score',
        'passing_options_score',
        'space_availability_score',
        'decision_difficulty_score'
    ]
].corr()

,receiver_pressure_score,passing_options_score,space_availability_score,decision_difficulty_score
receiver_pressure_score,1.000000,-0.309994,-0.936434,0.943594
passing_options_score,-0.309994,1.000000,0.274408,-0.581625
space_availability_score,-0.936434,0.274408,1.000000,-0.924435
decision_difficulty_score,0.943594,-0.581625,-0.924435,1.000000


<br>

Foram testadas diferentes combinações de ponderações para a construção do Decision Difficulty Score, procurando reduzir a influência excessiva de uma única componente e garantir uma representação mais equilibrada dos diferentes fatores considerados.

Após a análise dos resultados, optou-se pela configuração:

- **35% Receiver Pressure Score**
- **35% Passing Options Difficulty Score**
- **30% Space Difficulty Score**

Esta combinação reduz a dominância da componente de pressão observada nas versões iniciais da métrica, aumentando a contribuição das opções de passe e do espaço disponível para a avaliação da dificuldade de decisão.

A elevada correlação observada entre algumas métricas mantém-se expectável, uma vez que estas procuram capturar diferentes perspetivas do mesmo contexto espacial, não comprometendo, contudo, a utilidade da combinação de pesos selecionada.

****

<br>

## Resumo das Métricas Desenvolvidas

Durante este notebook foram desenvolvidas quatro métricas compostas com o objetivo de enriquecer os eventos de jogo com informação contextual proveniente dos dados StatsBomb 360.

   <br>
   
**1. Receiver Pressure Score** - procura quantificar o nível de pressão exercida sobre o jogador associado ao evento.

Componentes utilizadas:

**1.1. distância ao adversário mais próximo**

  - '≤' 2m → 100 pontos
  - '≤' 5m → 75 pontos
  - '≤' 10m → 50 pontos
  - '≤' 15m → 25 pontos
  - '>' 15m → 0 pontos
  
  <br>

**1.2. número de adversários presentes num raio de 5 metros**

  - 0 → 0 pontos
  - 1 → 25 pontos
  - 2 → 50 pontos
  - 3 → 75 pontos
  - ≥ 4 → 100 pontos
  
  <br>

**1.3. indicador de pressão disponibilizado pelo StatsBomb (`under_pressure`)**

  * False → 0 pontos
  * True → 100 pontos
   
  <br>


**Ponderações:**

* 50% distância ao adversário mais próximo
* 30% adversários até 5 metros
* 20% under pressure

<br>
---
<br>


**2. Passing Options Score** - procura quantificar a disponibilidade de soluções de passe para o jogador associado ao evento.

Componentes utilizadas:

**2.1. número de colegas de equipa disponíveis como opção de passe**

  - 0 → 0 pontos
  - 1 → 25 pontos
  - 2 → 50 pontos
  - 3 → 75 pontos
  - ≥ 4 → 100 pontos

  <br>

**2.2. distância ao adversário mais próximo**

  - '≤' 2m → 0 pontos
  - '≤' 5m → 25 pontos
  - '≤' 10m → 50 pontos
  - '≤' 15m → 75 pontos
  - '>' 15m → 100 pontos
  
   <br>

**2.3. número de adversários presentes num raio de 5 metros**

   <br>

**Ponderações:**

* 50% opções de passe
* 30% espaço disponível
* 20% adversários próximos

<br>
---
<br>

**3. Space Availability Score** - procura quantificar o espaço disponível em redor do jogador associado ao evento.

Componentes utilizadas:

**3.1. distância ao adversário mais próximo**

   <br>
   
**3.2. número de adversários presentes num raio de 5 metros**

   <br>
   
**3.3. número de adversários presentes num raio de 10 metros**

  * 0-1 → 100 pontos
  * 2-3 → 75 pontos
  * 4-5 → 50 pontos
  * 6-7 → 25 pontos
  * ≥ 8 → 0 pontos
  
   <br>

**Ponderações:**

* 40% espaço disponível
* 40% adversários próximos
* 20% adversários até 10 metros

   <br>
   <br>


---

**Decision Difficulty Score** - procura estimar a dificuldade associada à decisão tomada pelo jogador no momento do evento.

Componentes utilizadas:

* Receiver Pressure Score
* Passing Options Score (invertido)
* Space Availability Score (invertido)

Ponderações:

* 35% Receiver Pressure Score
* 35% Passing Options Score (invertido)
* 30% Space Availability Score (invertido)

   <br>
   <br>

Estas métricas procuram representar diferentes dimensões do contexto espacial dos eventos, permitindo complementar a análise tradicional baseada apenas nas ações realizadas pelos jogadores.


<br>

****

Chegada a esta fase do projeto, este é o estado atual do Modelo de Dados:

**DIM_MATCHES**

Primary Key - match_id

<br>

**DIM_PLAYERS**

Primary Key - player_id

<br>

**FCT_EVENTS**

Primary Key - id (event_id)

Foreign Keys - match_id, player_id, team_id

<br>

**FCT_FRAMES**

Foreign Key - id (event_id)

No entanto, para a construção de um melhor visual, eu já sei de antemão que vou precisar das métricas agregadas por jogador e equipa, para que dentro da visualização não tenha de fazer cálculos desnecessários.

Por isso, o próximo passo será construir uma FCT_PLAYER_STATS e uma FCT_TEAM_STATS com as métricas anteriormente calculadas, mas de forma agregada, e algumas outras novas.

<br>

---
## FCT_PLAYER_STATS
---

In [375]:
import numpy as np
import pandas as pd

# Excluir eventos da parte 5 (penaltis)
events_df_filtered = events_df.loc[events_df['match_half'] != 5].copy()

avg_cols = [
    'nearest_opponent_distance',
    'opponents_within_5m',
    'opponents_within_10m',
    'available_teammates',
    'nearest_opponent_distance_score',
    'opponents_within_5m_score',
    'under_pressure_score',
    'receiver_pressure_score',
    'available_teammates_score',
    'available_space_score',
    'nearby_opponents_score',
    'passing_options_score',
    'opponents_within_10m_space_score',
    'space_availability_score',
    'passing_options_difficulty_score',
    'space_difficulty_score',
    'decision_difficulty_score',
    'location_x',
    'location_y'
]

player_stats_df = (
    events_df_filtered
    .groupby('player_id')
    .apply(
        lambda g: pd.Series({
            # total ações
            'total_actions': len(g),

            # remates e xG
            'xG': g['shot_statsbomb_xg'].sum(),
            'shots': (g['type'] == 'Shot').sum(),
            'goals': (
                (g['type'] == 'Shot')
                & (g['shot_outcome'] == 'Goal')
            ).sum(),

            # passes e receções
            'receptions': (g['type'] == 'Ball Receipt*').sum(),
            'passes': (g['type'] == 'Pass').sum(),
            'successful_passes': (
                (g['type'] == 'Pass')
                & (
                    g['pass_outcome'].isna()
                    | g['pass_outcome'].eq('Complete')
                )
            ).sum(),

            # médias
            **{
                col: g[col].mean()
                for col in avg_cols
            }
        })
    )
    .reset_index()
)

# Renomear médias de localização
player_stats_df = player_stats_df.rename(columns={
    'location_x': 'avg_location_x',
    'location_y': 'avg_location_y'
})

# Métricas derivadas
player_stats_df['xG_difference'] = (
    player_stats_df['goals'] - player_stats_df['xG']
)

player_stats_df['successful_passes_perc'] = np.where(
    player_stats_df['passes'] > 0,
    player_stats_df['successful_passes'] / player_stats_df['passes'],
    np.nan
)

In [376]:
player_stats_df.head(5)

,player_id,total_actions,xG,shots,goals,receptions,passes,successful_passes,nearest_opponent_distance,opponents_within_5m,...,passing_options_score,opponents_within_10m_space_score,space_availability_score,passing_options_difficulty_score,space_difficulty_score,decision_difficulty_score,avg_location_x,avg_location_y,xG_difference,successful_passes_perc
0,10481,204.0,0.000000,0.0,0.0,45.0,69.0,56.0,5.390560,0.617647,...,57.528090,88.848039,64.578652,42.471910,35.421348,40.086376,59.884314,39.114216,0.000000,0.811594
1,11135,11.0,0.000000,0.0,0.0,1.0,2.0,1.0,3.507557,1.000000,...,61.388889,86.363636,52.222222,38.611111,47.777778,47.486111,64.700000,54.172727,0.000000,0.500000
2,11456,34.0,0.583520,4.0,0.0,11.0,7.0,5.0,3.193118,0.911765,...,44.629630,76.470588,52.222222,55.370370,47.777778,50.791667,93.835294,35.997059,-0.583520,0.714286
3,11990,41.0,0.023032,1.0,0.0,7.0,11.0,9.0,5.000773,0.463415,...,51.111111,85.975610,65.972222,48.888889,34.027778,40.857639,66.204878,26.800000,-0.023032,0.818182
4,16308,44.0,0.000000,0.0,0.0,9.0,15.0,15.0,6.513372,0.477273,...,60.069444,89.204545,67.638889,39.930556,32.361111,36.055556,50.250000,43.054545,0.000000,1.000000


<br>

---
## FCT_TEAM_STATS
---

In [377]:
import numpy as np
import pandas as pd

# Excluir eventos da parte 5 (penaltis)
events_df_filtered = events_df.loc[events_df['match_half'] != 5].copy()

avg_cols = [
    'nearest_opponent_distance',
    'opponents_within_5m',
    'opponents_within_10m',
    'available_teammates',
    'nearest_opponent_distance_score',
    'opponents_within_5m_score',
    'under_pressure_score',
    'receiver_pressure_score',
    'available_teammates_score',
    'available_space_score',
    'nearby_opponents_score',
    'passing_options_score',
    'opponents_within_10m_space_score',
    'space_availability_score',
    'passing_options_difficulty_score',
    'space_difficulty_score',
    'decision_difficulty_score'
]

team_stats_df = (
    events_df_filtered
    .groupby('team')
    .apply(
        lambda g: pd.Series({
            # total ações
            'total_actions': len(g),

            # remates e xG
            'xG': g['shot_statsbomb_xg'].sum(),
            'shots': (g['type'] == 'Shot').sum(),
            'goals': (
                (g['type'] == 'Shot')
                & (g['shot_outcome'] == 'Goal')
            ).sum(),

            # passes e receções
            'receptions': (g['type'] == 'Ball Receipt*').sum(),
            'passes': (g['type'] == 'Pass').sum(),
            'successful_passes': (
                (g['type'] == 'Pass')
                & (
                    g['pass_outcome'].isna()
                    | g['pass_outcome'].eq('Complete')
                )
            ).sum(),

            # médias
            **{
                col: g[col].mean()
                for col in avg_cols
            }
        })
    )
    .reset_index()
)

# Renomear médias de localização
team_stats_df = team_stats_df.rename(columns={
    'location_x': 'avg_location_x',
    'location_y': 'avg_location_y'
})

# Métricas derivadas
team_stats_df['xG_difference'] = (
    team_stats_df['goals'] - team_stats_df['xG']
)

team_stats_df['successful_passes_perc'] = np.where(
    team_stats_df['passes'] > 0,
    team_stats_df['successful_passes'] / team_stats_df['passes'],
    np.nan
)

In [378]:
team_stats_df

,team,total_actions,xG,shots,goals,receptions,passes,successful_passes,nearest_opponent_distance,opponents_within_5m,...,available_space_score,nearby_opponents_score,passing_options_score,opponents_within_10m_space_score,space_availability_score,passing_options_difficulty_score,space_difficulty_score,decision_difficulty_score,xG_difference,successful_passes_perc
0,Argentina,2361.0,2.758306,20.0,3.0,625.0,693.0,560.0,5.818958,0.604405,...,36.259446,84.921643,50.924433,87.049979,64.249370,49.075567,35.750630,42.050189,0.241694,0.808081
1,France,2025.0,2.272618,10.0,3.0,489.0,570.0,434.0,6.677774,0.561481,...,40.067237,85.975309,51.080379,89.160494,66.399756,48.919621,33.600244,40.920614,0.727382,0.761404


****

## Revisão Datasets Finais

In [379]:
# validar dimensões

In [380]:
matches_df.shape

(64, 16)

In [381]:
players_df.shape

(50, 13)

In [382]:
events_df.shape

(4407, 58)

In [383]:
frames_df.shape

(60641, 9)

In [384]:
player_stats_df.shape

(34, 29)

In [385]:
team_stats_df.shape

(2, 27)

<br>

In [386]:
# validar colunas

In [387]:
matches_df.columns.tolist()

['match_id',
 'match_date',
 'kick_off',
 'competition',
 'season',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'match_status',
 'match_week',
 'competition_stage',
 'stadium',
 'referee',
 'home_managers',
 'away_managers']

In [388]:
players_df.columns.tolist()

['player_id',
 'player_name',
 'player_nickname',
 'jersey_number',
 'team',
 'position_id',
 'position',
 'position_order',
 'player_photo',
 'team_logo',
 'x',
 'y',
 'starter']

In [389]:
events_df.columns.tolist()

['match_id',
 'id',
 'related_events',
 'index',
 'player_id',
 'player',
 'position',
 'team_id',
 'team',
 'timestamp',
 'minute',
 'second',
 'match_half',
 'type',
 'off_camera',
 'duration',
 'location_x',
 'location_y',
 'under_pressure',
 'pass_aerial_won',
 'pass_angle',
 'pass_assisted_shot_id',
 'pass_body_part',
 'pass_cross',
 'pass_deflected',
 'pass_end_x',
 'pass_end_y',
 'pass_goal_assist',
 'pass_height',
 'pass_inswinging',
 'pass_length',
 'pass_outcome',
 'pass_outswinging',
 'pass_recipient',
 'pass_shot_assist',
 'pass_switch',
 'pass_technique',
 'pass_through_ball',
 'pass_type',
 'shot_outcome',
 'shot_statsbomb_xg',
 'nearest_opponent_distance',
 'opponents_within_5m',
 'opponents_within_10m',
 'available_teammates',
 'nearest_opponent_distance_score',
 'opponents_within_5m_score',
 'under_pressure_score',
 'receiver_pressure_score',
 'available_teammates_score',
 'available_space_score',
 'nearby_opponents_score',
 'passing_options_score',
 'opponents_within_

In [390]:
frames_df.columns.tolist()

['id',
 'match_id',
 'teammate',
 'actor',
 'keeper',
 'frame_location_x',
 'frame_location_y',
 'visible_area',
 'visible_area_points']

In [391]:
player_stats_df.columns.tolist()

['player_id',
 'total_actions',
 'xG',
 'shots',
 'goals',
 'receptions',
 'passes',
 'successful_passes',
 'nearest_opponent_distance',
 'opponents_within_5m',
 'opponents_within_10m',
 'available_teammates',
 'nearest_opponent_distance_score',
 'opponents_within_5m_score',
 'under_pressure_score',
 'receiver_pressure_score',
 'available_teammates_score',
 'available_space_score',
 'nearby_opponents_score',
 'passing_options_score',
 'opponents_within_10m_space_score',
 'space_availability_score',
 'passing_options_difficulty_score',
 'space_difficulty_score',
 'decision_difficulty_score',
 'avg_location_x',
 'avg_location_y',
 'xG_difference',
 'successful_passes_perc']

In [392]:
team_stats_df.columns.tolist()

['team',
 'total_actions',
 'xG',
 'shots',
 'goals',
 'receptions',
 'passes',
 'successful_passes',
 'nearest_opponent_distance',
 'opponents_within_5m',
 'opponents_within_10m',
 'available_teammates',
 'nearest_opponent_distance_score',
 'opponents_within_5m_score',
 'under_pressure_score',
 'receiver_pressure_score',
 'available_teammates_score',
 'available_space_score',
 'nearby_opponents_score',
 'passing_options_score',
 'opponents_within_10m_space_score',
 'space_availability_score',
 'passing_options_difficulty_score',
 'space_difficulty_score',
 'decision_difficulty_score',
 'xG_difference',
 'successful_passes_perc']

<br>

In [393]:
# confirmar métricas criadas

events_df[
[
    'receiver_pressure_score',
    'passing_options_score',
    'space_availability_score',
    'decision_difficulty_score'
]
].describe()

,receiver_pressure_score,passing_options_score,space_availability_score,decision_difficulty_score
count,3627.000000,3627.000000,3627.000000,3627.000000
mean,39.824235,50.980838,65.261924,41.516612
std,23.294155,15.441467,20.626904,16.556948
min,0.000000,5.000000,0.000000,4.375000
25%,25.000000,40.000000,50.000000,27.500000
50%,45.000000,50.000000,60.000000,40.375000
75%,57.500000,60.000000,80.000000,54.875000
max,100.000000,92.500000,100.000000,89.625000


****

**Exportar Datasets Finais**

A ideia inicial foi exportar em csv para manter o critério. O problema é que alguns formatos de colunas (numeros decimais passavam para texto) ficavam desajustados e criavam grandes problemas.

Os datasets serão exportados em Excel, e são os que serão usados em PowerBI.

In [394]:
import os

os.makedirs(
    'data/final',
    exist_ok=True
)

matches_df.to_excel(
    'data/final/dim_matches.xlsx',
    index=False
)

players_df.to_excel(
    'data/final/dim_players.xlsx',
    index=False
)

events_df.to_excel(
    'data/final/fct_events.xlsx',
    index=False
)

frames_df.to_excel(
    'data/final/fct_frames.xlsx',
    index=False
)

player_stats_df.to_excel(
    'data/final/fct_player_stats.xlsx',
    index=False
)

team_stats_df.to_excel(
    'data/final/fct_team_stats.xlsx',
    index=False
)

print("Datasets finais exportados com sucesso.")

Datasets finais exportados com sucesso.
